# Comparison & Results Aggregation (`compare_results.ipynb`)

Implements **Section 12.3** of the Generative Models Benchmark Specification:
- Reads **only** the JSON files in `results/`. Never trains or samples.
- Generates Tables **T1–T6** and Figures **F1–F6**.
- Applies the **Section 10.3 meaningfulness rule**: a difference between two methods is called meaningful only if the two seed values of one method do not overlap with the other's **and** the gap is larger than the sampling-noise standard deviation from Block G.

In [ ]:
# 1. Colab setup cell (identical in every notebook per Section 3 & Section 12)
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = 'https://github.com/alisamadian/generative-models-benchmark.git'
REPO_DIR = Path('/content/generative-models-benchmark')
DRIVE_REPO_DIR = Path('/content/drive/MyDrive/generative-models-benchmark')

if 'google.colab' in sys.modules:
    from google.colab import drive
    if not Path('/content/drive/MyDrive').exists():
        drive.mount('/content/drive')
    if DRIVE_REPO_DIR.exists() and (DRIVE_REPO_DIR / 'common').exists():
        os.chdir(DRIVE_REPO_DIR)
    else:
        if not REPO_DIR.exists():
            subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
        else:
            subprocess.run(['git', '-C', str(REPO_DIR), 'pull'], check=False)
        os.chdir(REPO_DIR)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
else:
    # Local execution: ensure working directory is the repository root
    if Path.cwd().name == 'notebooks':
        os.chdir(Path.cwd().parent)
if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))

# Reload common.* modules if already imported in this Python kernel after git pull
import importlib
for _mod_name in list(sys.modules.keys()):
    if _mod_name == 'common' or _mod_name.startswith('common.'):
        importlib.reload(sys.modules[_mod_name])

import torch
from common.config import SPLIT_SEED, protocol_dict, protocol_hash
from common.utils import library_versions, mount_drive, set_seed

DRIVE_PATHS = mount_drive()
set_seed(SPLIT_SEED)

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f'GPU: {gpu_name} ({gpu_mem_gb:.2f} GB VRAM)')
else:
    print('GPU: None (running on CPU)')

print('Working directory:', Path.cwd())
print('Library versions:', library_versions())
print('Drive/Cache paths:', {k: str(v) for k, v in DRIVE_PATHS.items()})

## 1. Load All Results JSONs & Build Tables T1–T6 and Figures F1–F3, F6

In [ ]:
import pandas as pd
from IPython.display import Image, display
from pathlib import Path
from common.runner import build_comparison_tables_and_figures, check_meaningful_difference, load_all_results

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

tables = build_comparison_tables_and_figures(
    results_dir='./results',
    figures_dir='./figures',
    show_plots=True,
)

## 2. Table T1: Main Headline Table per Dataset (Both Seeds + Mean + Ceiling & Floor)

In [ ]:
for ds in ('mnist', 'cifar10'):
    print(f'\n=== Table T1 ({ds.upper()}): Headline Quality, Diversity & Reference Baselines ===')
    if not tables['T1'].empty:
        display(tables['T1'][tables['T1']['dataset'] == ds])
    else:
        print('No final results JSONs found yet in results/final/.')

## 3. Table T2: Efficiency Table (Parameters, Training Time & Sampling Throughput)

In [ ]:
print('=== Table T2: Model Complexity, Training Cost & Sampling Throughput ===')
display(tables['T2'])

## 4. Table T3: Class Balance Table (Evaluation Classifier Metrics)

In [ ]:
print('=== Table T3: Class Balance (TV Distance, Normalized Entropy & Mean Max Softmax) ===')
display(tables['T3'])

## 5. Table T4: Memorisation Table (Nearest-Neighbour Distances vs Test Control)

In [ ]:
print('=== Table T4: Memorisation Check (Inception NN Distance vs Test Control) ===')
display(tables['T4'])

## 6. Table T5: Hyperparameter Tuning Appendix (All 4 Trials per Method)

In [ ]:
print('=== Table T5: Validation-Only Hyperparameter Tuning Trials (Section 9) ===')
display(tables['T5'])

## 7. Table T6: Sampling-Noise Study (Block G) & Section 10.3 Meaningfulness Analysis

In [ ]:
print('=== Table T6: Sampling-Noise Study Across Generation Seeds (100, 101, 102) ===')
display(tables['T6'])

# Apply Section 10.3 meaningfulness rule between Diffusion and Flow Matching at NFE 20 and 100
if not tables['T1'].empty and not tables['T6'].empty:
    for ds in ('mnist', 'cifar10'):
        t1_ds = tables['T1'][tables['T1']['dataset'] == ds]
        t6_ds = tables['T6'][tables['T6']['dataset'] == ds]
        noise_std = float(t6_ds['fid_std'].max()) if not t6_ds.empty else 0.0
        for nfe in (20, 100):
            r_diff = t1_ds[(t1_ds['model'] == 'diffusion') & (t1_ds['sampler_nfe'] == f'ddim_nfe{nfe}')]
            r_fm = t1_ds[(t1_ds['model'] == 'flow_matching') & (t1_ds['sampler_nfe'] == f'euler_nfe{nfe}')]
            if not r_diff.empty and not r_fm.empty:
                s_diff = [float(r_diff.iloc[0]['fid_train_s0']), float(r_diff.iloc[0]['fid_train_s1'])]
                s_fm = [float(r_fm.iloc[0]['fid_train_s0']), float(r_fm.iloc[0]['fid_train_s1'])]
                verdict = check_meaningful_difference(s_diff, s_fm, noise_std)
                print(f'[{ds.upper()} First-Order NFE={nfe}] DDIM={s_diff} vs Euler={s_fm} -> {verdict["verdict"]}')

## 8. Figures F4 (Fixed-Seed Sample Grids) & F5 (Memorisation Nearest-Neighbour Grids)

In [ ]:
for fig_path in sorted(Path('figures').glob('f[45]_*.png')):
    print(f'Displaying {fig_path.name}:')
    display(Image(filename=str(fig_path)))